# RESULTADOS 2025 — raw → trusted

Contrato: `../docs/contrato_resultados_2025.md`. Dez campos, todos os registros. Este notebook chama funções reutilizáveis; não calcula indicadores finais nem aplica filtros de presença. Use **ENEM 2025 (.venv)** e execute todas as células após reiniciar o kernel. A execução completa substitui somente o Parquet derivado após validação; o raw permanece intacto.

In [1]:
from pathlib import Path
import sys

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/trusted_resultados.py").is_file() and (p / "raw").is_dir())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from src.trusted_resultados import executar, recursos
import pandas as pd

print("Projeto:", raiz)
print("Python:", sys.executable)
recursos(raiz)

Projeto: C:\Users\SamuelCaetanoPacheco\Desktop\ENEM-Data-Analysis
Python: C:\Users\SamuelCaetanoPacheco\Desktop\ENEM-Data-Analysis\.venv\Scripts\python.exe


{'ram_total_bytes': 8445833216,
 'ram_disponivel_bytes': 1328586752,
 'disco_livre_bytes': 490681901056}

## 1. Validar uma amostra

Primeiros 10 mil registros: teste do fluxo, sem inferência populacional ou publicação. Erros de contrato interrompem a execução.

In [2]:
amostra = executar(raiz, limite=10_000, memoria="256MB", threads=1)
assert amostra["registros_entrada"] == amostra["validacao"]["registros"]
print("Amostra validada:", amostra["registros_entrada"], "registros")

Amostra validada: 10000 registros


## 2. Executar a base completa

Leitura Latin-1 com `;`; staging em disco e SQL DuckDB. Mantém ausentes, eliminados, nulos e zeros. Publicação somente após contagens, chave, ano, esquema, releitura exata e hash do original conferidos. Limite DuckDB não representa teto de memória de todo o processo.

In [3]:
completo = executar(raiz, memoria="256MB", threads=1)
assert completo["publicado"]
assert completo["registros_entrada"] == completo["releitura"]["registros"]
assert completo["sha256_raw_antes"] == completo["sha256_raw_depois"]
print("Entrada / saída:", completo["registros_entrada"], completo["releitura"]["registros"])
print("Parquet:", completo["parquet"], "bytes:", completo["parquet_bytes"])
print("Tempo (s):", completo["segundos"])

Entrada / saída: 4810772 4810772
Parquet: trusted/resultados_2025_base.parquet bytes: 54733745
Tempo (s): 40.767


## 3. Conferir o relatório de qualidade

Pandas recebe apenas contagens agregadas pequenas. Achados de coerência são reportados sem corrigir ou excluir registros. Os JSON em `reports/` contêm também esquema, recursos, hashes e categorias.

In [4]:
v = completo["validacao"]
display(pd.DataFrame.from_dict(v["nulos"], orient="index", columns=["nulos"]))
display(pd.DataFrame({a: {k: n for k, n in r.items() if k != "presencas"}
                      for a, r in v["por_area"].items()}).T)
print("Chaves duplicadas:", v["chaves_duplicadas"])
print("Anos:", v["anos"])
print("Falhas de conversão:", completo["falhas_conversao"])
print("Categorias inesperadas:", completo["categorias_inesperadas"])

,nulos
NU_SEQUENCIAL,0
NU_ANO,0
TP_PRESENCA_CN,0
TP_PRESENCA_CH,0
TP_PRESENCA_LC,0
TP_PRESENCA_MT,0
NU_NOTA_CN,1550436
NU_NOTA_CH,1353217
NU_NOTA_LC,1353217
NU_NOTA_MT,1550436


,presente_sem_nota,ausente_com_nota,eliminado_com_nota,presenca_nula_com_nota,nota_zero,nota_negativa
CN,0,0,0,0,775,0
CH,0,0,0,0,9087,0
LC,0,0,0,0,2361,0
MT,0,0,0,0,893,0


Chaves duplicadas: 0
Anos: [{'ano': 2025, 'quantidade': 4810772}]
Falhas de conversão: {'NU_ANO': 0, 'TP_PRESENCA_CN': 0, 'TP_PRESENCA_CH': 0, 'TP_PRESENCA_LC': 0, 'TP_PRESENCA_MT': 0, 'NU_NOTA_CN': 0, 'NU_NOTA_CH': 0, 'NU_NOTA_LC': 0, 'NU_NOTA_MT': 0}
Categorias inesperadas: {'TP_PRESENCA_CN': [], 'TP_PRESENCA_CH': [], 'TP_PRESENCA_LC': [], 'TP_PRESENCA_MT': []}


## Próximo incremento

Definir contrato dos indicadores (filtros, população elegível e denominadores por pergunta), depois produzir tabela e gráficos. A trusted não contém filtro universal de presença. PARTICIPANTES continua independente; não há vínculo individual renda–nota.